# Topic 4: Unsupervised Learning - Customer Segmentation
**Module 1 - Introduction to Machine Learning in Python**


In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt
import seaborn as sns


## 1. k-Means Customer Segmentation


In [ ]:
np.random.seed(42)
n = 3000
df = pd.DataFrame({
    'avg_balance': np.random.lognormal(8, 1, n).round(0),
    'txn_count_monthly': np.random.poisson(15, n),
    'avg_txn_amount': np.random.lognormal(4, 1, n).round(2),
    'months_on_book': np.random.randint(6, 240, n),
    'num_products': np.random.choice([1,2,3,4,5], n, p=[0.3,0.3,0.2,0.15,0.05]),
})

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)


## 2. Elbow Method and Silhouette Analysis


In [ ]:
inertias = []
silhouettes = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(K_range, inertias, 'o-', color='steelblue')
ax1.set_xlabel('k'); ax1.set_ylabel('Inertia'); ax1.set_title('Elbow Method')
ax2.plot(K_range, silhouettes, 'o-', color='indianred')
ax2.set_xlabel('k'); ax2.set_ylabel('Silhouette Score'); ax2.set_title('Silhouette Analysis')
plt.tight_layout()
plt.show()


## 3. Fit Final Model and Profile Clusters


In [ ]:
optimal_k = 4
km = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['segment'] = km.fit_predict(X_scaled)

profile = df.groupby('segment').mean().round(1)
profile['count'] = df.groupby('segment').size()
print('Cluster Profiles:')
print(profile)
print()

# Business labels
labels = {0: 'High-Value Active', 1: 'New/Small', 2: 'Dormant', 3: 'Moderate'}
for seg, label in labels.items():
    count = (df['segment'] == seg).sum()
    print(f'  Segment {seg} ({label}): {count} customers')


## 4. PCA Visualization of Clusters


In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df['segment'], cmap='Set2', alpha=0.5, s=15)
plt.colorbar(scatter, label='Segment')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('Customer Segments (PCA Projection)')
plt.tight_layout()
plt.show()


## 5. Hierarchical Clustering with Dendrogram


In [ ]:
# Use a smaller sample for dendrogram readability
sample = X_scaled[:200]

Z = linkage(sample, method='ward')

plt.figure(figsize=(14, 6))
dendrogram(Z, truncate_mode='lastp', p=20, leaf_rotation=90, leaf_font_size=10)
plt.title('Hierarchical Clustering Dendrogram (Ward Linkage)')
plt.xlabel('Cluster Size')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()
